# 03 — Targeted transition dataset v1.1

Questo notebook costruisce il dataset diagnostico mirato per eventi dendritici, branching, recovery e stochasticità sinaptica. Il teacher Hay, i 642 segmenti, i `.mod`, i pesi, il burn-in, CVode, il boundary step da 1 ms e lo stato canonico restano invariati.

La decisione metodologica centrale è causale: `U_realized` viene osservato all'uscita del front-end sinaptico autentico, allo stesso timestamp del `NET_RECEIVE` e prima che la membrana avanzi sotto la nuova conduttanza. Non viene mai ricostruito da `S_(t+1)`.

## 1. Checkout riproducibile e teacher separato

In [ ]:
import importlib, json, os, shutil, subprocess, sys, zipfile
from pathlib import Path

ELM_REPOSITORY = "https://github.com/Zagred47/giada.git"
ELM_REF = os.environ.get("HAYFLOW_ELM_REF", "main")
TEACHER_REPOSITORY = "https://github.com/SelfishGene/neuron_as_deep_net.git"
TEACHER_COMMIT = "074c4666300a8ad246601dab179a97a6942f0f29"
NOTEBOOK_ROOT = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path.cwd().resolve()
WORKSPACE = NOTEBOOK_ROOT / "hayflow_workspace"
WORKSPACE.mkdir(parents=True, exist_ok=True)

def run(command, cwd=None):
    print("+", " ".join(map(str, command)), flush=True)
    subprocess.run(list(map(str, command)), cwd=cwd, check=True)

elm_override = os.environ.get("HAYFLOW_ELM_REPO")
mounted = [Path(elm_override).expanduser()] if elm_override else []
mounted.extend([Path.cwd(), *Path.cwd().parents])
ELM_REPO = next((p.resolve() for p in mounted if (p / "src" / "hayflow_teacher").is_dir()), None)
if ELM_REPO is None:
    ELM_REPO = WORKSPACE / "elmneuron"
    if not (ELM_REPO / ".git").is_dir():
        run(["git", "clone", ELM_REPOSITORY, ELM_REPO])
    run(["git", "fetch", "origin", ELM_REF], cwd=ELM_REPO)
    run(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=ELM_REPO)
TEACHER_REPO = Path(os.environ.get("HAYFLOW_TEACHER_REPO", ELM_REPO.parent / "neuron_as_deep_net")).expanduser().resolve()
if not (TEACHER_REPO / ".git").is_dir():
    run(["git", "clone", TEACHER_REPOSITORY, TEACHER_REPO])
run(["git", "checkout", "--detach", TEACHER_COMMIT], cwd=TEACHER_REPO)
assert subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=TEACHER_REPO, text=True).strip() == TEACHER_COMMIT
os.environ["HAYFLOW_ELM_REPO"] = str(ELM_REPO)
os.environ["HAYFLOW_TEACHER_REPO"] = str(TEACHER_REPO)
print("Owned repository:", ELM_REPO)
print("Canonical teacher:", TEACHER_REPO)

## 2. Dipendenze e compilazione dei MOD originali

In [ ]:
run([sys.executable, "-m", "pip", "install", "--quiet", "neuron==8.2.7", "numpy", "pandas", "matplotlib", "h5py", "pyarrow", "pyyaml"])
SIMULATION_DIR = TEACHER_REPO / "L5PC_NEURON_simulation"
compiled = list(SIMULATION_DIR.rglob("libnrnmech.so"))
if not compiled:
    nrnivmodl = shutil.which("nrnivmodl") or str(Path(sys.executable).parent / "nrnivmodl")
    run([nrnivmodl, "mods"], cwd=SIMULATION_DIR)
assert list(SIMULATION_DIR.rglob("libnrnmech.so")), "MOD compilation failed"
sys.path.insert(0, str(ELM_REPO))
for name in tuple(sys.modules):
    if name == "src.hayflow_teacher" or name.startswith("src.hayflow_teacher.") or name == "src.hayflow_data" or name.startswith("src.hayflow_data."):
        sys.modules.pop(name, None)
importlib.invalidate_caches()
print("Teacher mechanisms compiled without source changes.")

## 3. Calibrazione dendritica 01b come provenienza immutabile

In [ ]:
calibration_override = os.environ.get("HAYFLOW_CALIBRATION_SOURCE")
calibration_candidates = [Path(calibration_override).expanduser()] if calibration_override else []
calibration_candidates.extend([
    Path("/kaggle/input/datasets/alessandrobelli/hayflow-dendritic-protocol-calibration/hayflow_dendritic_protocol_calibration"),
    NOTEBOOK_ROOT / "hayflow_dendritic_protocol_calibration",
    NOTEBOOK_ROOT / "hayflow_dendritic_protocol_calibration.zip",
])
if Path("/kaggle/input").is_dir():
    calibration_candidates.extend(Path("/kaggle/input").rglob("hayflow_dendritic_protocol_calibration.zip"))
    calibration_candidates.extend(path.parent for path in Path("/kaggle/input").rglob("selected_dendritic_protocols.json"))
CALIBRATION_SOURCE = next((path.resolve() for path in calibration_candidates if path.exists()), None)
assert CALIBRATION_SOURCE is not None, "Calibrazione 01b non trovata. Imposta HAYFLOW_CALIBRATION_SOURCE."
print("Calibration source:", CALIBRATION_SOURCE)

## 4. Manifest, contratto 1.1 e burn-in misurato

In [ ]:
import yaml
from IPython.display import display
from src.hayflow_data import BurnInCriteria
from src.hayflow_teacher import TargetedDiagnosticDatasetSession, expected_audit_hashes

BASE_CONFIG_PATH = ELM_REPO / "configs" / "hayflow" / "transition_dataset_diagnostic.yml"
V1_CONFIG_PATH = ELM_REPO / "configs" / "hayflow" / "diagnostic_dataset_v1.yml"
TARGET_CONFIG_PATH = ELM_REPO / "configs" / "hayflow" / "targeted_transition_dataset_v1_1.yml"
base_config = yaml.safe_load(BASE_CONFIG_PATH.read_text(encoding="utf-8"))
target_config = yaml.safe_load(TARGET_CONFIG_PATH.read_text(encoding="utf-8"))
OUTPUT_DIR = NOTEBOOK_ROOT / "artifacts" / "diagnostic_dataset_v1_1"
if OUTPUT_DIR.exists():
    assert os.environ.get("HAYFLOW_RESET_OUTPUT") == "1", f"Output già presente: {OUTPUT_DIR}. Imposta HAYFLOW_RESET_OUTPUT=1 per rigenerarlo."
    shutil.rmtree(OUTPUT_DIR)
session = TargetedDiagnosticDatasetSession(
    ELM_REPO, TEACHER_REPO,
    calibration_source=CALIBRATION_SOURCE,
    dataset_config_path=TARGET_CONFIG_PATH,
    output_dir=OUTPUT_DIR,
    seed=base_config["runtime"]["seed"],
    expected_teacher_hashes=expected_audit_hashes(),
    native_snapshot_stride=target_config["storage"]["native_snapshot_stride_ms"],
)
teacher_report = session.prepare_teacher()
contract_report = session.prepare_targeted_contract()
burnin_config = dict(base_config["burnin"])
burnin_config["slow_mechanisms"] = tuple(burnin_config["slow_mechanisms"])
criteria = BurnInCriteria(**burnin_config)
burnin_report = session.run_burn_in(criteria)
current_smoke = session.run_somatic_current_smoke_test(**base_config["somatic_current"]["smoke_test"])
candidate_currents = base_config["somatic_current"]["calibration"]["candidate_amplitudes_na"]
session.calibrate_somatic_spike_current(candidate_amplitudes_na=candidate_currents)
session.calibrate_somatic_single_spike_current(candidate_amplitudes_na=candidate_currents)
display({"teacher": teacher_report, "contract": contract_report, "burnin": burnin_report})
assert teacher_report["segment_count"] == 642
assert contract_report["core_state_width"] == 17220
assert contract_report["privileged_state_width"] == 9182
assert burnin_report["converged"] and current_smoke["valid"]

## 5. Pilot causale del rilascio

Questa è la cella metodologicamente decisiva. Verifica stesso snapshot + stesso Random123 + stesso schedule ⇒ stesso release outcome e stessa transizione. Controlla inoltre che il campione Random123 previsto e il salto diretto degli stati sinaptici concordino.

In [ ]:
release_pilot = session.run_causal_release_pilot(
    transition_count=target_config["release_contract"]["pilot_transition_count"]
)
display(release_pilot)
assert release_pilot["valid"]
assert not release_pilot["future_state_used"]
assert not release_pilot["instrumentation_changes_teacher_dynamics"]

## 6. Pilot biologico adattivo

Lo sweep cerca separatamente spike assonale/somatico, bAP, Ca spike, NMDA spike breve e plateau. Stampa progresso ed ETA. Se una classe non ha positivi robusti e hard-negative su tutti i seed, si ferma qui.

In [ ]:
biological_pilot = session.run_targeted_biological_pilot(target_config["biological_pilot"])
display(biological_pilot)
assert biological_pilot["valid"]

## 7. Piano bilanciato per episodi indipendenti e split specializzati

In [ ]:
from src.hayflow_data import build_balanced_episode_plan, append_specialized_test_episodes, summarize_independent_support
protocols, planned_episodes = build_balanced_episode_plan(session.targeted_recipe_catalog)
protocols, planned_episodes = append_specialized_test_episodes(protocols, planned_episodes, session.targeted_recipe_catalog)
planned_transitions = sum(row.duration_ms for row in protocols)
planned_support = summarize_independent_support(planned_episodes)
snapshot_bank_report = session.prepare_snapshot_bank(protocols, conditioning_ms=4)
plan_report = session.accept_targeted_protocol_plan(protocols, pilot_report=biological_pilot)
display({"plan": plan_report, "support": planned_support, "snapshot_bank": snapshot_bank_report})
assert plan_report["valid"]

## 8. Generazione completa

Questa è la cella lunga. Il tracker mostra transizioni completate, traiettoria corrente ed ETA. Il dataset rimane diagnostico: 10–30 mila transizioni.

In [ ]:
dataset_manifest = session.generate_dataset(protocols)
display({k: dataset_manifest[k] for k in ("schema_version", "trajectory_count", "transition_count", "event_count", "size_estimate")})

## 9. Validazione esaustiva e report finale

In [ ]:
validation_report = session.validate_dataset_v1_1()
dataset_card = json.loads((OUTPUT_DIR / "dataset_card.json").read_text(encoding="utf-8"))
display({"validation": validation_report, "dataset_card": dataset_card})
assert validation_report["valid"]

## 10. Download ZIP con Base64 + Blob

Questa è la procedura di download stabile usata nel progetto: `make_archive`, trasferimento Base64 al browser e vero link temporaneo `<a download>`. Per artefatti molto grandi la codifica richiede memoria aggiuntiva nel kernel e nel browser.

In [ ]:
from base64 import b64encode
from shutil import make_archive
from IPython.display import Javascript, display

artifact_dir = OUTPUT_DIR.resolve()
zip_base = NOTEBOOK_ROOT / "hayflow_targeted_transition_dataset_v1_1"
zip_path = Path(make_archive(str(zip_base), "zip", root_dir=artifact_dir.parent, base_dir=artifact_dir.name))
payload = b64encode(zip_path.read_bytes()).decode("ascii")
filename = zip_path.name
print(f"ZIP pronto: {zip_path} ({zip_path.stat().st_size / 1024**3:.2f} GiB)")
display(Javascript(f'''
(() => {{
  const binary = atob("{payload}");
  const bytes = new Uint8Array(binary.length);
  for (let i = 0; i < binary.length; i++) bytes[i] = binary.charCodeAt(i);
  const blob = new Blob([bytes], {{type: "application/zip"}});
  const url = URL.createObjectURL(blob);
  const link = document.createElement("a");
  link.href = url;
  link.download = "{filename}";
  document.body.appendChild(link);
  link.click();
  link.remove();
  setTimeout(() => URL.revokeObjectURL(url), 60000);
}})();
'''))